## 📦 Cell 1 — Install Dependencies
Run once per Colab session. `edge-tts` added for AI voiceover.

In [ ]:
!apt-get update -qq
!apt-get install imagemagick nodejs -y -qq
!pip install moviepy==1.0.3 requests edge-tts nest_asyncio -q
!sed -i '/<policy domain="path" rights="none" pattern="@\*"\/>/d' /etc/ImageMagick-6/policy.xml
!sed -i 's/<policy domain="resource" name="width" value=".*"\/>/  <policy domain="resource" name="width" value="32KP"\/>/g' /etc/ImageMagick-6/policy.xml
!sed -i 's/<policy domain="resource" name="height" value=".*"\/>/  <policy domain="resource" name="height" value="32KP"\/>/g' /etc/ImageMagick-6/policy.xml
!sed -i 's/<policy domain="resource" name="area" value=".*"\/>/  <policy domain="resource" name="area" value="2GiB"\/>/g' /etc/ImageMagick-6/policy.xml
!sed -i 's/<policy domain="resource" name="disk" value=".*"\/>/  <policy domain="resource" name="disk" value="8GiB"\/>/g' /etc/ImageMagick-6/policy.xml
print('✅ Ready.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package libfftw3-double3:amd64.
Preparing to unpack .../01-libfftw3-double3_3.3.8-2ubuntu8_amd64.deb ...
Unpacking libfftw3-double3:amd64 (3.3.8-2ubuntu8) ...
Selecting previously unselected package liblqr-1-0:amd64.
Preparing to unpack .../02-liblqr-1-0_0.4.2-2.1_amd64.deb ...
Unpacking liblqr-1-0:amd64 (0.4.2-2.1) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../03-imagemagick-6-common_8%3a6.9.11.60+dfsg-1.3ubuntu0.

## 🎬 Cell 2 — Main Script

In [ ]:
import os
import re
import random
import asyncio
import requests
import numpy as np
import nest_asyncio
import edge_tts
from google.colab import userdata
from moviepy.editor import (
    VideoFileClip, TextClip, CompositeVideoClip,
    ColorClip, AudioFileClip, CompositeAudioClip
)
from moviepy.video.fx.all import fadein, crop
from moviepy.audio.fx.all import audio_fadeout, audio_loop

nest_asyncio.apply()

# ==========================================
# CONFIGURATION
# ==========================================
try:
    PEXELS_API_KEY = userdata.get('PEXELS_API_KEY')
except Exception:
    PEXELS_API_KEY = None
    print('⚠️  PEXELS_API_KEY nahi mili.')

try:
    JAMENDO_CLIENT_ID = userdata.get('JAMENDO_CLIENT_ID')
except Exception:
    JAMENDO_CLIENT_ID = None
    print('⚠️  JAMENDO_CLIENT_ID nahi mili — SoundHelix fallback.')

TARGET_W, TARGET_H = 1080, 1920
SAFE_BOTTOM        = 300
TTS_VOICE          = 'en-US-ChristopherNeural'

SOUNDHELIX_TRACKS = {
    'phonk':     'https://www.soundhelix.com/examples/mp3/SoundHelix-Song-3.mp3',
    'synthwave': 'https://www.soundhelix.com/examples/mp3/SoundHelix-Song-6.mp3',
    'lofi':      'https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3',
    'default':   'https://www.soundhelix.com/examples/mp3/SoundHelix-Song-2.mp3',
}
JAMENDO_TAGS = {
    'phonk':     'dark+hiphop',
    'synthwave': 'electronic+synthwave',
    'lofi':      'lofi+chill',
    'default':   'ambient+relaxing',
}

# ==========================================
# 1. VIDEO FETCHING (PEXELS)
# ==========================================
def get_pexels_video(query='dark aesthetic landscape', fallback_queries=None):
    fallback_queries = fallback_queries or ['dark moody background', 'abstract dark loop']
    for q in [query] + fallback_queries:
        print(f"➤ Pexels: '{q}'...")
        try:
            r = requests.get(
                f'https://api.pexels.com/videos/search?query={q}&per_page=5&orientation=portrait&size=medium',
                headers={'Authorization': PEXELS_API_KEY}, timeout=15
            )
            r.raise_for_status()
            videos = r.json().get('videos', [])
            if not videos:
                continue
            video      = random.choice(videos)
            candidates = [f for f in video['video_files'] if f.get('height', 0) >= f.get('width', 1)]
            best       = max(candidates, key=lambda f: f.get('height', 0)) if candidates else video['video_files'][0]
            data       = requests.get(best['link'], timeout=30)
            with open('bg_video.mp4', 'wb') as f:
                f.write(data.content)
            print('   ✓ Video ready.')
            return 'bg_video.mp4'
        except Exception as e:
            print(f'   ⚠️ {e}')
    return None

# ==========================================
# 2. BACKGROUND MUSIC (JAMENDO → SOUNDHELIX)
# ==========================================
def _download_direct(url, output_path):
    try:
        r = requests.get(url, timeout=30, stream=True)
        r.raise_for_status()
        with open(output_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
        return True
    except Exception as e:
        print(f'   ⚠️ Download failed: {e}')
        return False

def get_background_music(mood_key='lofi', output_path='bg_music.mp3'):
    if os.path.exists(output_path):
        os.remove(output_path)
    if JAMENDO_CLIENT_ID:
        tags = JAMENDO_TAGS.get(mood_key, JAMENDO_TAGS['default'])
        print(f"➤ Jamendo: '{tags}'...")
        try:
            r       = requests.get(
                f'https://api.jamendo.com/v3.0/tracks/?client_id={JAMENDO_CLIENT_ID}'
                f'&format=json&limit=20&tags={tags}&audioformat=mp32&order=popularity_total',
                timeout=15
            )
            results = r.json().get('results', [])
            if results:
                track = random.choice(results)
                if track.get('audio') and _download_direct(track['audio'], output_path):
                    print(f"   ✓ '{track.get('name')}'")
                    return output_path
        except Exception as e:
            print(f'   ⚠️ Jamendo error: {e}')
    fallback_url = SOUNDHELIX_TRACKS.get(mood_key, SOUNDHELIX_TRACKS['default'])
    print('➤ SoundHelix fallback...')
    if _download_direct(fallback_url, output_path):
        print('   ✓ SoundHelix ready.')
        return output_path
    return None

# ==========================================
# 3. AI VOICEOVER (EDGE-TTS) — QUOTE ONLY
# Author name is NOT spoken — visual only.
# ==========================================
def _fmt_srt(seconds):
    ms = int((seconds % 1) * 1000)
    s  = int(seconds) % 60
    m  = (int(seconds) // 60) % 60
    h  = int(seconds) // 3600
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def _ticks_to_sec(ticks):
    return ticks / 10_000_000

async def _tts_async(text, voice, audio_path, srt_path):
    communicate     = edge_tts.Communicate(text, voice)
    word_boundaries = []
    with open(audio_path, 'wb') as f:
        async for chunk in communicate.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                word_boundaries.append(chunk)

    print(f'   ✓ WordBoundary events: {len(word_boundaries)}')
    srt_lines = []

    if word_boundaries:
        # Method 1: exact timings from edge-tts
        for idx, wb in enumerate(word_boundaries, 1):
            start = _ticks_to_sec(wb['audio_offset'])
            end   = _ticks_to_sec(wb['audio_offset'] + wb.get('duration', 4_000_000))
            word  = wb.get('text', '').strip()
            if word:
                srt_lines.append(f"{idx}\n{_fmt_srt(start)} --> {_fmt_srt(end)}\n{word}\n")
    else:
        # Method 2: proportional estimation by character count
        print('   ⚠️ Estimating timings from audio duration...')
        audio_dur   = AudioFileClip(audio_path).duration
        raw_words   = text.split()
        start_pad   = 0.3
        available   = max(audio_dur - start_pad - 0.4, 0.5)
        char_counts = [max(len(w.strip('.,—\"\u201c\u201d')), 1) for w in raw_words]
        total_chars = sum(char_counts)
        cur         = start_pad
        for idx, (word, chars) in enumerate(zip(raw_words, char_counts), 1):
            dur = max((chars / total_chars) * available, 0.12)
            srt_lines.append(f"{idx}\n{_fmt_srt(cur)} --> {_fmt_srt(cur + dur)}\n{word}\n")
            cur += dur
        print(f'   ✓ Estimated {len(raw_words)} word timings.')

    with open(srt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(srt_lines))
    print(f'   ✓ SRT written: {len(srt_lines)} entries.')

def generate_voiceover(quote_text,
                        voice=TTS_VOICE,
                        audio_path='voiceover.mp3',
                        srt_path='voiceover.srt'):
    print(f'➤ Edge-TTS voiceover (quote only, no author)...')
    for f in [audio_path, srt_path]:
        if os.path.exists(f):
            os.remove(f)
    try:
        asyncio.get_event_loop().run_until_complete(
            _tts_async(quote_text.strip(), voice, audio_path, srt_path)
        )
    except Exception as e:
        print(f'   ⚠️ TTS fail: {e}')
        return None, None
    if not os.path.exists(audio_path):
        return None, None
    dur = AudioFileClip(audio_path).duration
    print(f'   ✓ Voiceover ready! {dur:.1f}s')
    return audio_path, srt_path if os.path.exists(srt_path) else None

# ==========================================
# 4. SRT PARSER
# ==========================================
def _srt_to_sec(t):
    t       = t.strip().replace(',', '.')
    h, m, s = t.split(':')
    s, ms   = s.split('.')
    return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000

def _parse_srt(srt_path):
    with open(srt_path, encoding='utf-8') as f:
        content = f.read().replace('\r\n', '\n').replace('\r', '\n')
    pattern = re.compile(
        r'\d+\n(\d{2}:\d{2}:\d{2}[,.]\d{3}) --> (\d{2}:\d{2}:\d{2}[,.]\d{3})\n(.+)',
        re.MULTILINE
    )
    words = []
    for m in pattern.finditer(content):
        word = m.group(3).strip()
        if word:
            words.append((_srt_to_sec(m.group(1)), _srt_to_sec(m.group(2)), word))
    print(f'   ✓ SRT parsed: {len(words)} words.')
    return words

# ==========================================
# 5. WORD REVEAL ANIMATION
# Top-to-bottom ink fill — no sliding, no bouncing.
# Each word stays in place and gets exposed top→bottom.
# ==========================================
def _make_reveal_fn(reveal_dur):
    """
    Factory function — avoids Python closure bug inside loops.
    Returns a MoviePy fl() compatible function.
    clip internal time t=0 → fully hidden
    clip internal time t=reveal_dur → fully visible
    """
    def reveal(get_frame, t):
        frame      = get_frame(t)
        h          = frame.shape[0]
        ratio      = min(t / reveal_dur, 1.0) if reveal_dur > 0 else 1.0
        reveal_px  = int(ratio * h)
        result     = np.zeros_like(frame)
        if reveal_px > 0:
            result[:reveal_px] = frame[:reveal_px]
        return result
    return reveal

def _measure_word_width(word, font, fontsize):
    """Render word invisibly just to get its pixel width."""
    tc = TextClip(word, fontsize=fontsize, color='white', font=font)
    w  = tc.w
    tc.reader.close() if hasattr(tc, 'reader') else None
    return w

def create_word_reveal_clips(srt_path, total_duration, font,
                              words_per_line=5,
                              fontsize=64,
                              word_gap=20,
                              reveal_dur=0.30,
                              stagger=0.16):
    """
    Builds per-word top-to-bottom reveal clips.

    Layout logic:
      - words_per_line words are grouped into a line
      - line appears at line_start time (from SRT)
      - within the line each word is staggered by `stagger` seconds
      - each word reveals top→bottom over `reveal_dur` seconds
      - entire line disappears when the next line begins
    """
    words = _parse_srt(srt_path)
    if not words:
        print('   ⚠️ No words — skipping reveal.')
        return []

    # Group into lines
    lines = []
    for i in range(0, len(words), words_per_line):
        chunk      = words[i : i + words_per_line]
        line_words = [w[2] for w in chunk]
        line_start = chunk[0][0]
        line_end   = words[i + words_per_line][0] if (i + words_per_line) < len(words) else total_duration
        lines.append((line_start, line_end, line_words))

    center_y  = int(TARGET_H * 0.46)
    all_clips = []

    print(f'   ✓ {len(lines)} lines to render:')

    for line_idx, (line_start, line_end, line_words) in enumerate(lines):
        line_dur = max(line_end - line_start, 0.5)

        # ── Step 1: measure word widths to calculate x positions ──
        widths     = [_measure_word_width(w, font, fontsize) for w in line_words]
        total_w    = sum(widths) + word_gap * (len(widths) - 1)
        start_x    = (TARGET_W - total_w) // 2  # center the line horizontally

        line_preview = '  '.join(line_words)
        print(f'      Line {line_idx+1} [{line_start:.1f}s]: {line_preview}')

        # ── Step 2: create reveal clip per word ───────────────────
        current_x = start_x
        for i, (word, word_w) in enumerate(zip(line_words, widths)):
            word_start = line_start + i * stagger
            word_dur   = max(line_end - word_start, reveal_dur + 0.05)

            # Shadow — slightly offset, dark, same reveal
            shadow = (TextClip(word, fontsize=fontsize, color='black', font=font)
                      .fl(_make_reveal_fn(reveal_dur))
                      .set_opacity(0.65)
                      .set_start(word_start)
                      .set_duration(word_dur)
                      .set_position((current_x + 3, center_y + 5)))

            # Main word
            main = (TextClip(word, fontsize=fontsize, color='white', font=font,
                             stroke_color='black', stroke_width=2)
                    .fl(_make_reveal_fn(reveal_dur))
                    .set_start(word_start)
                    .set_duration(word_dur)
                    .set_position((current_x, center_y)))

            all_clips.extend([shadow, main])
            current_x += word_w + word_gap

    total_word_clips = len(all_clips) // 2  # each word = shadow + main
    print(f'   ✓ {total_word_clips} word reveal clips ready.')
    return all_clips

# ==========================================
# 6. HELPERS
# ==========================================
def safe_font(preferred='Poppins-Bold'):
    try:
        TextClip('test', font=preferred, fontsize=10)
        return preferred
    except Exception:
        print(f"   ⚠️ '{preferred}' nahi mila → Arial-Bold.")
        return 'Arial-Bold'

def crop_to_portrait(clip, target_w=TARGET_W, target_h=TARGET_H):
    target_ratio = target_w / target_h
    clip_ratio   = clip.w / clip.h
    clip = clip.resize(height=target_h) if clip_ratio > target_ratio else clip.resize(width=target_w)
    return crop(clip, width=target_w, height=target_h,
                x_center=clip.w / 2, y_center=clip.h / 2)

def make_text_with_shadow(text, fontsize, color, font,
                           size=None, method='label', align='center'):
    shadow = (TextClip(text, fontsize=fontsize, color='black', font=font,
                       size=size, method=method, align=align).set_opacity(0.55))
    main   = TextClip(text, fontsize=fontsize, color=color, font=font,
                      stroke_color='black', stroke_width=1.5,
                      size=size, method=method, align=align)
    return main, shadow

def loop_or_trim(audio_clip, duration):
    if audio_clip.duration < duration:
        return audio_loop(audio_clip, duration=duration)
    return audio_clip.subclip(0, duration)

# ==========================================
# 7. THE REEL DESIGNER
# ==========================================
def design_reel(quote_text, author_name, category,
                min_duration=9, tts_voice=TTS_VOICE):
    print('\n🎬 Designing the Reel...')

    video_query = 'nature dark aesthetic'
    fallback_q  = ['dark abstract background', 'moody cinematic loop']
    mood_key    = 'lofi'

    if 'Hustle' in category:
        video_query = 'gym motivation dark'
        fallback_q  = ['workout silhouette dark', 'fitness dark cinematic']
        mood_key    = 'phonk'
    elif 'Tech' in category:
        video_query = 'cyberpunk computer'
        fallback_q  = ['futuristic dark tech', 'digital abstract dark']
        mood_key    = 'synthwave'

    bg_video_path            = get_pexels_video(video_query, fallback_q)
    bg_music_path            = get_background_music(mood_key)
    voiceover_path, srt_path = generate_voiceover(quote_text, tts_voice)  # no author_name

    if not bg_video_path or not bg_music_path:
        print('❌ Video/music missing.')
        return None

    # Duration driven by TTS
    if voiceover_path:
        vo_probe     = AudioFileClip(voiceover_path)
        tts_duration = vo_probe.duration + 1.5
        vo_probe.close()
    else:
        tts_duration = 0
    target_duration = max(min_duration, tts_duration)
    print(f'   ℹ️  Duration: {target_duration:.1f}s')

    # Background video
    raw_clip = VideoFileClip(bg_video_path)
    if raw_clip.duration < target_duration:
        from moviepy.video.fx.all import loop as video_loop
        raw_clip = video_loop(raw_clip, duration=target_duration)
    bg_clip = crop_to_portrait(raw_clip.subclip(0, target_duration))

    # Music
    music_clip = loop_or_trim(AudioFileClip(bg_music_path), target_duration)
    music_clip = audio_fadeout(music_clip.volumex(0.12), 1.5)

    # Voiceover
    if voiceover_path:
        final_audio = CompositeAudioClip([
            music_clip,
            AudioFileClip(voiceover_path).set_start(0.5)
        ])
    else:
        final_audio = music_clip
    bg_clip = bg_clip.set_audio(final_audio)

    # Dark overlay
    dark_overlay = (ColorClip(size=(TARGET_W, TARGET_H), color=(0, 0, 0))
                    .set_opacity(0.50).set_duration(target_duration))

    font = safe_font('Poppins-Bold')

    # ── Word-by-word top-to-bottom reveal ─────────────────────────
    print('\n📝 Building word reveal clips...')
    text_clips = []
    if srt_path and os.path.exists(srt_path):
        text_clips = create_word_reveal_clips(
            srt_path, target_duration, font,
            words_per_line=5,
            fontsize=64,
            word_gap=22,
            reveal_dur=0.30,
            stagger=0.16
        )

    if not text_clips:
        # Fallback: simple fade-in static text
        print('   → Fallback: static quote.')
        q_main, q_shadow = make_text_with_shadow(
            f'"{quote_text}"', fontsize=52, color='white',
            font=font, size=(900, None), method='caption'
        )
        text_clips = [
            q_shadow.set_position(('center', int(TARGET_H*0.46)+4)).set_duration(target_duration).fx(fadein, 1.0),
            q_main.set_position(('center', int(TARGET_H*0.46))).set_duration(target_duration).fx(fadein, 1.0)
        ]

    # ── Author name — visual only, slower fade-in ─────────────────
    spaced_author = '  '.join(f'— {author_name}'.upper())
    author_main, author_shadow = make_text_with_shadow(
        spaced_author, fontsize=30, color='#C9A84C', font=font
    )
    author_y = TARGET_H - SAFE_BOTTOM
    author_main   = (author_main
                     .set_position(('center', author_y))
                     .set_duration(target_duration)
                     .fx(fadein, 3.0))  # slow fade — appears after quote starts
    author_shadow = (author_shadow
                     .set_position(('center', author_y + 3))
                     .set_duration(target_duration)
                     .fx(fadein, 3.0))

    # ── Compose ───────────────────────────────────────────────────
    print('\n🎞️  Rendering...')
    final_video = CompositeVideoClip(
        [bg_clip, dark_overlay]
        + text_clips
        + [author_shadow, author_main],
        size=(TARGET_W, TARGET_H)
    )
    final_video.write_videofile(
        'final_viral_reel.mp4',
        fps=30, threads=4,
        codec='libx264', audio_codec='aac'
    )

    for f in [bg_video_path, bg_music_path, voiceover_path, srt_path]:
        if f and os.path.exists(f):
            os.remove(f)

    print('\n🔥 Reel ready! Files panel mein check karein.')
    return 'final_viral_reel.mp4'

# ==========================================
# TEST RUN
# ==========================================
test_quote    = "As had been the case formerly with tyranny justified by theology, and is the case now with tyranny justified by therapy — the oppressor succeeds not only in subduing his victim but also in robbing him of a vocabulary for articulating his victimization."
test_author   = 'Thomas Szasz'
test_category = 'Philosophy'

design_reel(test_quote, test_author, test_category)

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':




🎬 Designing the Reel...
➤ Pexels: 'nature dark aesthetic'...
   ✓ Video ready.
➤ Jamendo: 'lofi+chill'...
   ✓ 'InfinityX'
➤ Edge-TTS voiceover (quote only, no author)...
   ✓ WordBoundary events: 0
   ⚠️ Estimating timings from audio duration...
   ✓ Estimated 43 word timings.
   ✓ SRT written: 43 entries.
   ✓ Voiceover ready! 14.9s
   ℹ️  Duration: 16.4s

📝 Building word reveal clips...
   ✓ SRT parsed: 43 words.
   ✓ 9 lines to render:
      Line 1 [0.3s]: As  had  been  the  case
      Line 2 [1.4s]: formerly  with  tyranny  justified  by
      Line 3 [3.4s]: theology,  and  is  the  case
      Line 4 [4.8s]: now  with  tyranny  justified  by
      Line 5 [6.5s]: therapy  —  the  oppressor  succeeds
      Line 6 [8.5s]: not  only  in  subduing  his
      Line 7 [9.8s]: victim  but  also  in  robbing
      Line 8 [11.3s]: him  of  a  vocabulary  for
      Line 9 [12.7s]: articulating  his  victimization.
   ✓ 43 word reveal clips ready.

🎞️  Rendering...
Moviepy - Building video f

MoviePy - Done.
Moviepy - Writing video final_viral_reel.mp4



Moviepy - Done !
Moviepy - video ready final_viral_reel.mp4

🔥 Reel ready! Files panel mein check karein.


'final_viral_reel.mp4'